# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第11章 範疇知覚とCue Weighting ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `sim_identification_data.R`

対象章: 第8章「知覚実験・刺激作成・PsychoPy」・第11章「範疇知覚と Cue Weighting」

目的:
  実データを収録するまでの間、本書の作例（第11章「発展課題」直前の
  通しの例）で使う同定課題データを、現実的なパラメータでシミュレートする。
  このスクリプトが生成する data/processed/identification_sim.csv は
  **実測データではなくシミュレーションである**ことを、本文・ファイル名の
  両方で明示する。

刺激設計（第8章 §刺激連続体の合成 と対応）:
  scripts/praat/make_vot_continuum.praat と同じ設計思想で、
  VOT 0ms から 80ms まで 10ms 刻みの9段階の連続体を想定する。

個人差のモデル化（第11章 §重みの推定方法と個人差 と対応）:
  各被験者は、母集団の平均境界・平均傾きのまわりに個人差を持つ
  （境界の個人差 SD=8ms、傾きの個人差はロジット尺度で変動）。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_subjects <- 24
steps      <- seq(0, 80, by = 10)          # 9段階（第8章の刺激連続体と対応）
n_reps     <- 12                            # 1被験者・1ステップあたりの繰り返し数

pop_boundary_mean <- 32     # 母集団平均境界（VOT ms）
pop_boundary_sd    <- 8      # 境界の被験者間SD
pop_slope_mean     <- 0.22   # 母集団平均傾き（ロジット/ms）
pop_slope_sd        <- 0.07  # 傾きの被験者間SD（負の値にならないようlogでモデル化）

subj <- tibble(
  subject  = sprintf("s%02d", 1:n_subjects),
  boundary = rnorm(n_subjects, pop_boundary_mean, pop_boundary_sd),
  slope    = exp(rnorm(n_subjects, log(pop_slope_mean), pop_slope_sd / pop_slope_mean))
)

d <- subj |>
  tidyr::crossing(step = steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p = slope * (step - boundary),
    p_resp  = plogis(logit_p),
    response_bin = rbinom(n(), 1, p_resp),
    response = if_else(response_bin == 1, "p", "b")
  ) |>
  select(subject, step, rep, response)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "identification_sim.csv"))

cat("生成したシミュレーションデータ:", nrow(d), "行 /", n_subjects, "名 x",
    length(steps), "ステップ x", n_reps, "反復\n")
cat("母集団パラメータ: 境界平均", pop_boundary_mean, "ms (SD", pop_boundary_sd, "),",
    "傾き平均", pop_slope_mean, "(幾何SD要因", round(exp(pop_slope_sd/pop_slope_mean),2), ")\n")


## `ch11_worked_example.R`

対象章: 第11章「範疇知覚と Cue Weighting」通しの作例

本スクリプトは、第8章で作成したVOT連続体（9段階、0-80ms）に対する
同定課題の反応を、シミュレーションデータ（scripts/R/sim_identification_data.R
が生成する data/processed/identification_sim.csv）に対して実行する。
**実測データではない。** 実データが得られ次第、同じコードをそのまま適用できる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

sim_path <- here::here("data", "processed", "identification_sim.csv")
if (!file.exists(sim_path)) {
  source(here::here("scripts", "R", "sim_identification_data.R"))
}

d <- read_csv(sim_path, show_col_types = FALSE) |>
  mutate(
    subject  = factor(subject),
    response = as.integer(response == "p"),
    step_c   = as.numeric(scale(step, scale = FALSE))
  )

m <- glmer(response ~ step_c + (1 + step_c | subject),
           data = d, family = binomial,
           control = glmerControl(optimizer = "bobyqa",
                                  optCtrl = list(maxfun = 2e5)))

fx <- fixef(m)
step_mean <- mean(d$step)
boundary_pop <- step_mean - fx["(Intercept)"] / fx["step_c"]

co <- coef(m)$subject
per_subj <- tibble(
  subject  = rownames(co),
  slope    = co[, "step_c"],
  boundary = step_mean - co[, "(Intercept)"] / co[, "step_c"]
)

cat("=== 固定効果（母集団平均） ===\n")
print(summary(m)$coefficients)
cat("\n母集団平均のカテゴリー境界（50%点）:", round(boundary_pop, 1), "ms\n")
cat("母集団平均の傾き（ロジット/ms）:", round(fx["step_c"], 3), "\n\n")

cat("=== 被験者ごとの境界・傾き（要約） ===\n")
print(summary(per_subj[c("boundary", "slope")]))
cat("境界の被験者間SD:", round(sd(per_subj$boundary), 1), "ms\n")
cat("傾きの被験者間SD:", round(sd(per_subj$slope), 3), "\n\n")

isSingular_flag <- isSingular(m)
cat("特異適合か:", isSingular_flag, "\n")

# 図: 観測された反応割合 + モデルの当てはめ曲線
obs <- d |>
  group_by(step) |>
  summarise(prop = mean(response), n = n(), .groups = "drop")

curve_df <- tibble(step = seq(min(d$step), max(d$step), length.out = 200)) |>
  mutate(prop = plogis(fx["(Intercept)"] + fx["step_c"] * (step - step_mean)))

p <- ggplot(obs, aes(step, prop)) +
  geom_line(data = curve_df, linewidth = 0.9, colour = "#34518A") +
  geom_point(aes(size = n), colour = "#1B2333") +
  geom_vline(xintercept = as.numeric(boundary_pop), linetype = 2, colour = "#A8681F") +
  scale_y_continuous(labels = scales::percent, limits = c(0, 1)) +
  scale_size_continuous(guide = "none") +
  labs(x = "刺激ステップ (VOT, ms)", y = "「/p/」反応の割合",
       caption = "scripts/R/ch11_worked_example.R より生成（シミュレーションデータ）")
save_fig(p, "fig11-01-identification-curve", chapter = "ch11")

# 図: 被験者ごとの境界・傾きの分布（個人差）
p2 <- ggplot(per_subj, aes(boundary, slope)) +
  geom_point(size = 2, colour = "#34518A", alpha = 0.75) +
  labs(x = "被験者ごとの境界 (VOT, ms)", y = "被験者ごとの傾き（ロジット/ms）",
       caption = "scripts/R/ch11_worked_example.R より生成（シミュレーションデータ）")
save_fig(p2, "fig11-02-boundary-slope-individual", chapter = "ch11")

write_csv(per_subj, here::here("data", "processed", "ch11_per_subject_estimates.csv"))
cat("\n完了。図は assets/figures/ch11/ に、被験者ごとの推定値は data/processed/ch11_per_subject_estimates.csv に保存した。\n")


## `ch11_cue_weighting.R`

対象章: 第11章「範疇知覚とCue Weighting」通しの作例(2手がかり版)

「重みの推定方法と個人差」節の作例。scripts/R/sim_identification_data.R
の単一手がかり(VOT)の設計を、VOT・F0の立ち上がりという2つの手がかりに
拡張する。実測データではない。各被験者にVOTへの依存度とF0への依存度の
個人差を与え、glmerで交互作用モデルを当てはめたうえで、被験者ごとの
手がかり重み(ランダム傾き)を抽出し、重み空間上の個人差を図示する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260905)

n_subjects <- 24
vot_steps  <- seq(0, 80, by = 20)     # 5段階
f0_steps   <- seq(-40, 40, by = 20)   # 5段階(基準からのHzの立ち上がり差)
n_reps     <- 6

pop_vot_weight <- 0.09    # VOTへの平均的な依存度(ロジット/ms)
pop_f0_weight  <- 0.035   # F0への平均的な依存度(ロジット/Hz)
boundary_vot   <- 40
boundary_f0    <- 0

subj <- tibble(
  subject = sprintf("s%02d", 1:n_subjects),
  # 個人差: VOTへの依存度とF0への依存度を、それぞれ独立な乱数として設定する。
  # 2つの重みのあいだに正・負いずれの相関があるかは、事前に仮定せず、
  # 推定モデルの結果から確認する。
  vot_weight = pmax(rnorm(n_subjects, pop_vot_weight, 0.03), 0.01),
  f0_weight  = pmax(rnorm(n_subjects, pop_f0_weight, 0.013), 0.002)
)

d <- subj |>
  tidyr::crossing(vot = vot_steps) |>
  tidyr::crossing(f0 = f0_steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p  = vot_weight * (vot - boundary_vot) + f0_weight * (f0 - boundary_f0),
    p_resp   = plogis(logit_p),
    response = rbinom(n(), 1, p_resp)
  ) |>
  select(subject, vot, f0, rep, response)

d <- d |> mutate(vot_c = as.numeric(scale(vot, scale = FALSE)),
                  f0_c  = as.numeric(scale(f0, scale = FALSE)))

m_cue <- glmer(response ~ vot_c * f0_c + (1 + vot_c + f0_c | subject),
               data = d, family = binomial,
               control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))

print(summary(m_cue)$coefficients)

co <- coef(m_cue)$subject
per_subj <- tibble(
  subject       = rownames(co),
  vot_weight_est = co[, "vot_c"],
  f0_weight_est  = co[, "f0_c"]
)

r_val <- cor(per_subj$vot_weight_est, per_subj$f0_weight_est)
cat("\nVOT重みとF0重みの被験者間相関: r =", round(r_val, 3), "\n")
cat("VOT重みの範囲:", round(range(per_subj$vot_weight_est), 3), "\n")
cat("F0重みの範囲:", round(range(per_subj$f0_weight_est), 3), "\n")

p <- ggplot(per_subj, aes(vot_weight_est, f0_weight_est)) +
  geom_point(size = 2.5, colour = "#34518A", alpha = 0.8) +
  geom_smooth(method = "lm", se = TRUE, colour = "#A8681F", linewidth = 0.7, formula = y ~ x) +
  labs(x = "VOTへの重み(ロジット/ms)", y = "F0への重み(ロジット/Hz)",
       caption = paste0("シミュレーションデータ(r = ", round(r_val, 2),
                         ")。scripts/R/ch11_cue_weighting.R より生成。"))
save_fig(p, "fig11-03-cue-weighting-space", chapter = "ch11")
cat("完了。図はassets/figures/ch11/に保存した。\n")
